# Wav2Vec2 Model. From HuggingFace Transformers to ONNX

## Evaluating Wav2Vec2 ONNX model

### Converting Wav2Vec2 Model to ONNX

In [ ]:
# Import libraries
from loguru import logger
from optimum.onnxruntime import ORTModelForAudioClassification
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

# -----------------------
# USER CONFIGURATION (edit these)
# -----------------------
MODEL_DIR = "/mnt/media/fair/audio/replay_attacks/modelos/Wav2Vec2_HF/1"  # <- put your trained model folder here
SAVE_DIR = "/mnt/media/fair/audio/replay_attacks/modelos/Wav2Vec2_HF/onnx/1_onnx"

# Evaluation routine
logger.info(f"Loading model from {MODEL_DIR} ...")
model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_DIR)
logger.success("Model loaded succesfully.")

logger.info(f"Loading feature extractor from {MODEL_DIR}...")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_DIR)
logger.success("Loaded feature extractor from model_dir.")

logger.info("Converting Wav2Vec2 model to ONNX...")
ort_model = ORTModelForAudioClassification.from_pretrained(
    MODEL_DIR,
    export=True,
    # provider="CUDAExecutionProvider"
    provider="CPUExecutionProvider"
)
ort_model.save_pretrained(SAVE_DIR)
logger.success("Wav2Vec2 model converted succesfully.")

2025-10-07 16:07:29.388 | INFO     | __main__:<module>:14 - Loading model from /mnt/media/fair/audio/replay_attacks/modelos/Wav2Vec2_HF/1 ...
2025-10-07 16:07:29.658 | SUCCESS  | __main__:<module>:16 - Model loaded succesfully.
2025-10-07 16:07:29.660 | INFO     | __main__:<module>:18 - Loading feature extractor from /mnt/media/fair/audio/replay_attacks/modelos/Wav2Vec2_HF/1...
2025-10-07 16:07:29.661 | SUCCESS  | __main__:<module>:20 - Loaded feature extractor from model_dir.
2025-10-07 16:07:29.662 | INFO     | __main__:<module>:22 - Converting Wav2Vec2 model to ONNX...
2025-10-07 16:07:36.880 | SUCCESS  | __main__:<module>:30 - Wav2Vec2 model converted succesfully.


### Audio processing

In [12]:
import librosa
from loguru import logger
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

# -----------------------
# USER CONFIGURATION (edit these)
# -----------------------
MODEL_DIR = "/mnt/media/fair/audio/replay_attacks/modelos/Wav2Vec2_HF/1"  # <- put your trained model folder here

logger.info(f"Loading feature extractor from {MODEL_DIR}...")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_DIR)
logger.success("Loaded feature extractor from model_dir.")

# Audio processing
# audio_path = "/home/pepelacasa/MM-PR-01568-audio_replay_attacks/audio/test_sine_16k_int16.wav"
# audio_path = "/mnt/media/fair/audio/replay_attacks/datasets/ASVSpoof2017/ASVspoof2017_V2_train/ASVspoof2017_V2_train/T_1000003.wav"
# audio_path = "/mnt/media/fair/audio/replay_attacks/datasets/ReMASC/core/train/data/1010206.wav"
audio_path = "/home/pepelacasa/MM-PR-01568-audio_replay_attacks/audio/PA_E_0061595.flac"

audio, sr = librosa.load(audio_path, sr=feature_extractor.sampling_rate)
inputs = feature_extractor(
    audio,
    sampling_rate=sr,
    return_tensors="pt",
    padding=False,
    truncation=False
)
print(inputs)

2025-10-17 13:54:27.271 | INFO     | __main__:<module>:10 - Loading feature extractor from /mnt/media/fair/audio/replay_attacks/modelos/Wav2Vec2_HF/1...
2025-10-17 13:54:27.273 | SUCCESS  | __main__:<module>:12 - Loaded feature extractor from model_dir.


{'input_values': tensor([[ 4.1822e-04, -4.2804e-04, -4.2804e-04,  ..., -4.9104e-06,
         -4.9104e-06, -4.9104e-06]])}


### ONNX model inference

In [ ]:
import torch
import numpy as np
from optimum.onnxruntime import ORTModelForAudioClassification

SAVE_DIR = "/mnt/media/fair/audio/replay_attacks/modelos/Wav2Vec2_HF/onnx/1_onnx"

logger.info("Loading ONNX saved model...")
onnx_model = ORTModelForAudioClassification.from_pretrained(SAVE_DIR)
logger.success("ONNX model loaded succesfully.")

# ONNX inference
with torch.no_grad():
    outputs = onnx_model(**inputs)
    logits = outputs.logits
    logits_cpu = logits.detach().cpu().numpy()

    if logits_cpu.ndim == 2 and logits_cpu.shape[1] > 1:
        exp = np.exp(logits_cpu - np.max(logits_cpu, axis=1, keepdims=True))
        probs = exp / np.sum(exp, axis=1, keepdims=True)
        scores = probs[:, 1]
        print(probs)
        print(scores)

2025-10-17 13:57:27.157 | INFO     | __main__:<module>:7 - Loading ONNX saved model...
2025-10-17 13:57:27.961 | SUCCESS  | __main__:<module>:9 - ONNX model loaded succesfully.


SequenceClassifierOutput(loss=None, logits=tensor([[-3.2715,  2.4685]]), hidden_states=None, attentions=None)
[[0.00320439 0.9967956 ]]
[0.9967956]


## Evaluating Wav2Vec2 trained model

In [11]:
import torch
import numpy as np
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

MODEL_DIR = "/mnt/media/fair/audio/replay_attacks/modelos/Wav2Vec2_HF/1"

logger.info(f"Loading model from {MODEL_DIR} ...")
model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_DIR)
logger.success("Model loaded succesfully.")

logger.info(f"Loading feature extractor from {MODEL_DIR}...")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_DIR)
logger.success("Loaded feature extractor from model_dir.")

# Audio processing
# audio_path = "/home/pepelacasa/MM-PR-01568-audio_replay_attacks/audio/test_sine_16k_int16.wav"
# audio_path = "/mnt/media/fair/audio/replay_attacks/datasets/ASVSpoof2017/ASVspoof2017_V2_train/ASVspoof2017_V2_train/T_1000003.wav"
audio_path = "/mnt/media/fair/audio/replay_attacks/datasets/ReMASC/core/train/data/1010206.wav"

audio, sr = librosa.load(audio_path, sr=feature_extractor.sampling_rate)
inputs = feature_extractor(
    audio,
    sampling_rate=sr,
    return_tensors="pt",
    padding=False,
    truncation=False
)
input_values = inputs["input_values"]
attention_mask = inputs.get("attention_mask", None)
if attention_mask is None:
    attention_mask = torch.ones_like(input_values, dtype=torch.long)
else:
    attention_mask = attention_mask.squeeze(0)

# Wav2Vec2 model inference
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)
model.to(device)
input_values = input_values.to(device)
attention_mask = attention_mask.to(device)

logger.info("Evaluating model...")
outputs = model(input_values=input_values, attention_mask=attention_mask)
logits = outputs.logits
logits_cpu = logits.detach().cpu().numpy()

if logits_cpu.ndim == 2 and logits_cpu.shape[1] > 1:
    exp = np.exp(logits_cpu - np.max(logits_cpu, axis=1, keepdims=True))
    probs = exp / np.sum(exp, axis=1, keepdims=True)
    scores = probs[:, 1]
    print(scores)

2025-10-08 07:41:11.139 | INFO     | __main__:<module>:7 - Loading model from /mnt/media/fair/audio/replay_attacks/modelos/Wav2Vec2_HF/1 ...


2025-10-08 07:41:11.405 | SUCCESS  | __main__:<module>:9 - Model loaded succesfully.
2025-10-08 07:41:11.405 | INFO     | __main__:<module>:11 - Loading feature extractor from /mnt/media/fair/audio/replay_attacks/modelos/Wav2Vec2_HF/1...
2025-10-08 07:41:11.406 | SUCCESS  | __main__:<module>:13 - Loaded feature extractor from model_dir.
2025-10-08 07:41:11.554 | INFO     | __main__:<module>:42 - Evaluating model...


[0.00623515]
